## 데이터 탐색

### 데이터셋 구성

- 

---

# 0️⃣ 프로젝트 개요 및 환경 설정

In [30]:
# uv 프로젝트 환경이 있으면 uv sync로 의존성 설치 (권장)
#!pip install numpy pandas matplotlib seaborn scipy scikit-learn statsmodels pingouin scikit_posthocs xgboost -q
print("pip 설치 완료!")

pip 설치 완료!


In [31]:
%pip install scikit-posthocs

Note: you may need to restart the kernel to use updated packages.


c:\Users\610ha\Desktop\study\Third-project\.venv\Scripts\python.exe: No module named pip


In [32]:
# ============================================================
# 라이브러리 Import
# ============================================================

# 데이터 처리 및 분석
import os
import numpy as np
import pandas as pd
from datetime import datetime, timedelta
import warnings
import ast

# 시각화
import matplotlib.pyplot as plt
import seaborn as sns

# 통계 분석
from scipy import stats
from scipy.stats import shapiro, levene, ttest_ind, chi2_contingency, f_oneway
from scipy.stats import mannwhitneyu, fisher_exact, kruskal
from scipy.stats import skew, kurtosis
from statsmodels.stats.multicomp import pairwise_tukeyhsd, MultiComparison
import pingouin as pg


# 출력 설정
warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)

# 한글 폰트 설정
import platform
if platform.system() == 'Windows':
    plt.rcParams['font.family'] = 'Malgun Gothic'
elif platform.system() == 'Darwin':  # macOS
    plt.rcParams['font.family'] = 'AppleGothic'
else:  # Linux
    plt.rcParams['font.family'] = 'NanumGothic'

plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['figure.figsize'] = (12, 6)

# 참고: seed 고정으로 팀원 간 동일한 결과 재현 가능
np.random.seed(42)

print("="*60)
print("라이브러리 로드 완료!")
print("한글 폰트 설정 완료!")
print("="*60)

라이브러리 로드 완료!
한글 폰트 설정 완료!


# 데이터 로드

In [33]:
df_c = pd.read_csv("./tft_game_dataset/TFT_Challenger_MatchData.csv")
df_g = pd.read_csv("./tft_game_dataset/TFT_GrandMaster_MatchData.csv")
df_m = pd.read_csv("./tft_game_dataset/TFT_Master_MatchData.csv")
df_d = pd.read_csv("./tft_game_dataset/TFT_Diamond_MatchData.csv")
df_p = pd.read_csv("./tft_game_dataset/TFT_Platinum_MatchData.csv")

df_champion = pd.read_csv("./tft_game_dataset/TFT_Champion_CurrentVersion.csv")
df_item = pd.read_csv("./tft_game_dataset/TFT_Item_CurrentVersion.csv")

print(df_c.info())
print(df_g.info())
print(df_m.info())
print(df_d.info())
print(df_p.info())

print(df_champion.info())
print(df_item.info())

<class 'pandas.DataFrame'>
RangeIndex: 79999 entries, 0 to 79998
Data columns (total 8 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   gameId          79999 non-null  str    
 1   gameDuration    79999 non-null  float64
 2   level           79999 non-null  int64  
 3   lastRound       79999 non-null  int64  
 4   Ranked          79999 non-null  int64  
 5   ingameDuration  79999 non-null  float64
 6   combination     79999 non-null  str    
 7   champion        79999 non-null  str    
dtypes: float64(2), int64(3), str(3)
memory usage: 4.9 MB
None
<class 'pandas.DataFrame'>
RangeIndex: 80000 entries, 0 to 79999
Data columns (total 8 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   gameId          80000 non-null  str    
 1   gameDuration    80000 non-null  float64
 2   level           80000 non-null  int64  
 3   lastRound       80000 non-null  int64  
 4   Ranked          80

In [34]:
print(df_c.shape)
print(df_g.shape)
print(df_m.shape)
print(df_d.shape)
print(df_p.shape)
print(df_champion.shape)
print(df_item.shape)


(79999, 8)
(80000, 8)
(79999, 8)
(80000, 8)
(80000, 8)
(52, 12)
(54, 2)


In [35]:
df_c.head()

,gameId,gameDuration,level,lastRound,Ranked,ingameDuration,combination,champion
0,KR_4247538593,2142.470703,8,35,1,2134.272217,"{'DarkStar': 2, 'Protector': 4, 'Rebel': 1, 'S...","{'JarvanIV': {'items': [27], 'star': 3}, 'Sona..."
1,KR_4247538593,2142.470703,9,35,2,2134.272217,"{'Blaster': 2, 'Mercenary': 1, 'Rebel': 6, 'Se...","{'Malphite': {'items': [7], 'star': 2}, 'Yasuo..."
2,KR_4247538593,2142.470703,8,34,3,2073.459229,"{'Cybernetic': 1, 'DarkStar': 3, 'Demolitionis...","{'KaiSa': {'items': [99, 2, 23], 'star': 2}, '..."
3,KR_4247538593,2142.470703,8,33,4,1998.146729,"{'Blaster': 1, 'Cybernetic': 1, 'DarkStar': 1,...","{'KaiSa': {'items': [44, 37], 'star': 2}, 'Ann..."
4,KR_4247538593,2142.470703,9,33,5,1986.443237,"{'Blaster': 2, 'Demolitionist': 2, 'Mercenary'...","{'Ziggs': {'items': [], 'star': 1}, 'Yasuo': {..."


In [36]:
print(df_c.loc[0, 'combination'])
print(df_c.loc[0, 'champion'])

{'DarkStar': 2, 'Protector': 4, 'Rebel': 1, 'Set3_Celestial': 3, 'Set3_Mystic': 4, 'StarGuardian': 2, 'TemplateTrait': 1}
{'JarvanIV': {'items': [27], 'star': 3}, 'Sona': {'items': [46], 'star': 3}, 'Rakan': {'items': [37, 69], 'star': 3}, 'XinZhao': {'items': [69, 25, 25], 'star': 3}, 'Neeko': {'items': [], 'star': 2}, 'Karma': {'items': [], 'star': 2}, 'Soraka': {'items': [], 'star': 2}, 'Lulu': {'items': [59], 'star': 1}}


시즌 2와 시즌 3이 섞인 것 같다..하

In [37]:
# 각 csv match 파일 시너지 검증 시작..
dfs = {
    'challenger': df_c,
    'grandmaster': df_g,
    'master': df_m,
    'diamond': df_d,
    'platinum': df_p
}

In [38]:
#준환님이 정리해준 시즌2 시너지목록/ 일부수정(Set2_)
season2_traits = {
    "Steel", "Shadow", "Lunar", "Mountain", "Poison", "Ocean", "Wind", "Set2_Glacial",
    "Light", "Desert", "Woodland", "Electric", "Inferno", "Set2_Blademaster", "Berserker",
    "Druid", "Summoner", "Mystic", "Avatar", "Set2_Assassin", "Alchemist", "Soulbound",
    "Mage", "Set2_Ranger", "Warden", "Predator", "Crystal"
}

In [39]:
# 시즌2 시너지있는거 카운트00
def has_season2_trait(trait_list):
    return bool(set(trait_list) & season2_traits)

for name, df in dfs.items():
    df['combination'] = df['combination'].apply(ast.literal_eval)
    season2_mask = df['combination'].apply(has_season2_trait)  # 버그 수정
    count = season2_mask.sum()
    print(f"{name}: {count}")

challenger: 0
grandmaster: 0
master: 0
diamond: 416
platinum: 2814


In [40]:
# 다이아몬드 시너지 목록 확인
unique_d = set()

for x in df_d['combination']:
    unique_d.update(x.keys())

print(sorted(unique_d))

['Alchemist', 'Avatar', 'Berserker', 'Blaster', 'Celestial', 'Chrono', 'Crystal', 'Cybernetic', 'DarkStar', 'Demolitionist', 'Desert', 'Druid', 'Electric', 'Inferno', 'Infiltrator', 'Light', 'Mage', 'ManaReaver', 'MechPilot', 'Mercenary', 'Metal', 'Mountain', 'Mystic', 'Ocean', 'Poison', 'Predator', 'Protector', 'Rebel', 'Set2_Assassin', 'Set2_Blademaster', 'Set2_Glacial', 'Set2_Ranger', 'Set3_Blademaster', 'Set3_Brawler', 'Set3_Celestial', 'Set3_Mystic', 'Set3_Sorcerer', 'Set3_Void', 'Shadow', 'Sniper', 'Soulbound', 'SpacePirate', 'StarGuardian', 'Starship', 'Summoner', 'TemplateTrait', 'Valkyrie', 'Vanguard', 'Warden', 'Wind', 'Woodland']


In [41]:
# 플래티넘 시너지 목록 확인
unique_p = set()

for x in df_p['combination']:
    unique_p.update(x.keys())

print(sorted(unique_p))

['Alchemist', 'Avatar', 'Berserker', 'Blaster', 'Celestial', 'Chrono', 'Crystal', 'Cybernetic', 'DarkStar', 'Demolitionist', 'Desert', 'Druid', 'Electric', 'Inferno', 'Infiltrator', 'Light', 'Mage', 'ManaReaver', 'MechPilot', 'Mercenary', 'Metal', 'Mountain', 'Mystic', 'Ocean', 'Poison', 'Predator', 'Protector', 'Rebel', 'Set2_Assassin', 'Set2_Blademaster', 'Set2_Glacial', 'Set2_Ranger', 'Set3_Blademaster', 'Set3_Brawler', 'Set3_Celestial', 'Set3_Mystic', 'Set3_Sorcerer', 'Set3_Void', 'Shadow', 'Sniper', 'Soulbound', 'SpacePirate', 'StarGuardian', 'Starship', 'Summoner', 'TemplateTrait', 'Valkyrie', 'Vanguard', 'Warden', 'Wind', 'Woodland']


---
##

In [42]:
(df_c['gameDuration'] - df_c['ingameDuration']).describe()

count    79999.000000
mean       274.363092
std        248.500081
min          0.000000
25%          8.903381
50%        231.943481
75%        438.631531
max       2008.146240
dtype: float64